# 1. Inference: XLM-RoBERTa classifier + Qwen 14b + ViSoLex

In [1]:
# ============================================================
# Inference: XLM-RoBERTa classifier + Qwen 14b + ViSoLex
# Chạy trên Kaggle GPU
# ============================================================

# ============================================================
# 1. Cài thư viện
# ============================================================
!pip install transformers torch ollama

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [2]:
# ============================================================
# 2. Import
# ============================================================
import os
import re
import json
import subprocess
import time
import pickle

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
import ollama

/Users/nals_macbook_108/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/nals_macbook_108/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ============================================================
# 3. Cấu hình PATH — tự động detect Kaggle vs Local
# ============================================================
IS_KAGGLE = os.path.exists("/kaggle/working")

if IS_KAGGLE:
    DATA_PATH   = "/kaggle/input/multilex-norm-vi/"
    OUTPUT_PATH = "/kaggle/working/"
    MODEL_PATH  = "/kaggle/input/xlmr-classifier/xlmr_normalize_classifier"
else:
    DATA_PATH   = "../data/"
    OUTPUT_PATH = "../outputs/"
    MODEL_PATH  = "../models/xlmr_normalize_classifier"

train_path = DATA_PATH + "multilexnorm2026_vi_train.csv"
val_path   = DATA_PATH + "multilexnorm2026_vi_validation.csv"
test_path  = DATA_PATH + "multilexnorm2026_vi_test.csv"

os.makedirs(OUTPUT_PATH, exist_ok=True)

for path in [train_path, val_path, test_path]:
    print(path, "->", "FOUND" if os.path.exists(path) else "NOT FOUND")

print(f"\nMôi trường: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"DATA_PATH:   {DATA_PATH}")
print(f"OUTPUT_PATH: {OUTPUT_PATH}")
print(f"MODEL_PATH:  {MODEL_PATH}")

../data/multilexnorm2026_vi_train.csv -> FOUND
../data/multilexnorm2026_vi_validation.csv -> FOUND
../data/multilexnorm2026_vi_test.csv -> FOUND

Môi trường: Local
DATA_PATH:   ../data/
OUTPUT_PATH: ../outputs/
MODEL_PATH:  ../models/xlmr_normalize_classifier


In [4]:
# ============================================================
# 4. Hàm parse token list
# Dùng cùng hàm với 01/02/03
# ============================================================
def parse_token_list(text):
    text = str(text)
    tokens = re.findall(r"'([^']*)'", text)
    return tokens

In [5]:
# ============================================================
# 5. Đọc và parse dữ liệu
# ============================================================
train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)

for df in [train_df, val_df, test_df]:
    df["raw_tokens"]  = df["raw"].apply(parse_token_list)
    df["norm_tokens"] = df["norm"].apply(parse_token_list)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 8371 | Val: 1050 | Test: 522


In [6]:
# ============================================================
# 6. Detect device
# ============================================================
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Device: GPU - {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Device: M1 MPS")
else:
    device = torch.device("cpu")
    print("Device: CPU")

Device: M1 MPS


In [7]:
# ============================================================
# 7. Load model fine-tuned từ 04_finetune
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=True)
model     = AutoModelForTokenClassification.from_pretrained(MODEL_PATH)
model     = model.to(device)
model.eval()

print(f"Loaded model from {MODEL_PATH}")

Loaded model from ../models/xlmr_normalize_classifier


In [8]:
# ============================================================
# 8. Tạo label NEED_NORM/KEEP (để tính ERR sau này)
# ============================================================
def create_token_labels(row):
    return [1 if r != n else 0
            for r, n in zip(row["raw_tokens"], row["norm_tokens"])]

val_df["labels"] = val_df.apply(create_token_labels, axis=1)

In [9]:
# ============================================================
# 9. Chuẩn bị val_dataset để inference classifier
# ============================================================
def tokenize_and_align_labels(raw_tokens, word_labels, tokenizer, max_length=128):
    tokenized = tokenizer(
        raw_tokens,
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors="pt"
    )
    word_ids = tokenized.word_ids(0)
    aligned_labels = [
        -100 if word_idx is None else word_labels[word_idx]
        for word_idx in word_ids
    ]
    tokenized["labels"] = torch.tensor(aligned_labels)
    return tokenized

def prepare_dataset(df, tokenizer, max_length=128):
    all_input_ids, all_attention_masks, all_labels = [], [], []
    for _, row in df.iterrows():
        result = tokenize_and_align_labels(
            row["raw_tokens"], row["labels"], tokenizer, max_length
        )
        all_input_ids.append(result["input_ids"][0])
        all_attention_masks.append(result["attention_mask"][0])
        all_labels.append(result["labels"])
    return {
        "input_ids":      torch.stack(all_input_ids),
        "attention_mask": torch.stack(all_attention_masks),
        "labels":         torch.stack(all_labels)
    }

from torch.utils.data import Dataset

class NormDataset(Dataset):
    def __init__(self, data):
        self.input_ids      = data["input_ids"]
        self.attention_mask = data["attention_mask"]
        self.labels         = data["labels"]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels":         self.labels[idx]
        }

print("Tokenizing val set...")
val_data    = prepare_dataset(val_df, tokenizer)
val_dataset = NormDataset(val_data)
print(f"Val dataset: {len(val_dataset)}")

Tokenizing val set...
Val dataset: 1050


In [10]:

# ============================================================
# 10. Inference Classifier → classifier_preds
# ============================================================
all_predictions = []

with torch.no_grad():
    for i in range(len(val_dataset)):
        batch        = val_dataset[i]
        input_ids    = batch["input_ids"].unsqueeze(0).to(device)
        attention_mask = batch["attention_mask"].unsqueeze(0).to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds   = outputs.logits.argmax(-1)[0].cpu().tolist()
        all_predictions.append(preds)

print(f"Đã predict cho {len(all_predictions)} câu")

# Convert subword predictions → word predictions
def get_word_level_predictions(raw_tokens, subword_preds, tokenizer, max_length=128):
    tokenized = tokenizer(
        raw_tokens,
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors="pt"
    )
    word_ids   = tokenized.word_ids(0)
    word_preds = [None] * len(raw_tokens)

    for subword_idx, word_idx in enumerate(word_ids):
        if word_idx is not None and word_preds[word_idx] is None:
            word_preds[word_idx] = subword_preds[subword_idx]

    word_preds = [p if p is not None else 0 for p in word_preds]
    return word_preds

val_df["classifier_preds"] = [
    get_word_level_predictions(row["raw_tokens"], all_predictions[i], tokenizer)
    for i, row in val_df.iterrows()
]

print(val_df[["raw_tokens", "labels", "classifier_preds"]].head(3))

Đã predict cho 1050 câu
                                          raw_tokens  \
0  [Bên, ni, đèo, thì, "bả, sá",, bên, tê, đèo, t...   
1  [2, mày, bán, tao, ngồi, đếm, xiền, xem, có, đ...   
2  [Vứt, con, đấy, đi, hành, hung, ng, khác, tron...   

                                              labels  \
0               [0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]   
1  [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...   
2      [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]   

                                    classifier_preds  
0               [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  
1  [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...  
2      [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]  


In [11]:
# Save để dùng lại
val_df.to_pickle(OUTPUT_PATH + "val_with_classifier_preds.pkl")
print("Saved val_with_classifier_preds.pkl")

Saved val_with_classifier_preds.pkl


In [18]:
# ============================================================
# 11. Setup Ollama
# ============================================================
if IS_KAGGLE:
    os.system("apt-get install -y zstd pciutils lshw")
    os.system("curl -fsSL https://ollama.com/install.sh | sh")
    os.system("pkill -f ollama 2>/dev/null")
    time.sleep(2)
    process = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    time.sleep(5)
    os.system("ollama pull qwen2.5:14b")
else:
    # Local: Ollama đã cài sẵn, chỉ cần start nếu chưa chạy
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    time.sleep(3)
    print("Ollama started locally")
    # Local đã pull sẵn, không cần pull lại

ollama_client = ollama.Client(timeout=120)

Ollama started locally


In [20]:
!ollama pull qwen2.5:14b

]11;?\pulling manifest ⠋ pulling manifest ⠹ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pull

In [21]:
# Test
response = ollama_client.chat(
    model="qwen2.5:14b",
    messages=[{"role": "user", "content": "Test: 'mk' nghĩa là gì trong tiếng Việt?"}]
)
print(response["message"]["content"])

"MK" không phải là một cụm từ hoặc ký hiệu phổ biến trong tiếng Việt và có thể gây nhầm lẫn nếu không có ngữ cảnh cụ thể. Tuy nhiên, nó có thể liên quan đến các từ hoặc cụm từ nước ngoài được sử dụng trong ngữ cảnh trực tuyến hoặc chuyên ngành nhất định. Nếu bạn cung cấp thêm ngữ cảnh, tôi sẽ giúp làm rõ hơn về ý nghĩa của "MK" trong trường hợp đó.


In [22]:
# ============================================================
# 12. Load ViSoLex dictionary
# ============================================================
import requests

visolex_path = OUTPUT_PATH + "visolex_dict.json"

# Download nếu chưa có
if not os.path.exists(visolex_path):
    url = "https://raw.githubusercontent.com/HaDung2002/visolex/main/dict/dictionary.json"
    response = requests.get(url)
    with open(visolex_path, "w", encoding="utf-8") as f:
        f.write(response.text)
    print("Downloaded ViSoLex dictionary")
else:
    print("ViSoLex dictionary already exists, loading...")

with open(visolex_path, "r", encoding="utf-8") as f:
    visolex_dict = json.load(f)

simple_lookup = {}
multi_lookup  = {}

for nsw, data in visolex_dict.items():
    normalized_list = data["normalized"]
    if len(normalized_list) == 1:
        simple_lookup[nsw] = normalized_list[0]
    else:
        multi_lookup[nsw] = normalized_list

print(f"ViSoLex entries: {len(visolex_dict)}")
print(f"Simple lookup:   {len(simple_lookup)}")
print(f"Multi lookup:    {len(multi_lookup)}")

Downloaded ViSoLex dictionary
ViSoLex entries: 3091
Simple lookup:   2796
Multi lookup:    295


In [24]:
# ============================================================
# 13. Hardcode dictionary
# Token có 1 nghĩa áp đảo trong thực tế
# ============================================================
HARDCODE_DICT = {
    "r"   : "rồi",
    "h"   : "giờ",
    "v"   : "vậy",
    "s"   : "sao",
    "hong": "không",
    "rùi" : "rồi",
    "dị"  : "vậy",
}

In [25]:
# ============================================================
# 13b. Load high-confidence lookup từ output của 02 + 03
# ============================================================
import pandas as pd
from collections import defaultdict, Counter
import re

# Load token level results từ 02
token_results_df = pd.read_csv(OUTPUT_PATH + "val_lookup_token_level_results.csv")

# Load error analysis từ 03
wrong_token_meta_df = pd.read_csv(OUTPUT_PATH + "error_wrong_tokens_with_meta.csv")

# Rebuild mapping_counter từ train data
# (cần thiết để tính confidence)
def parse_token_list(text):
    text = str(text)
    tokens = re.findall(r"'([^']*)'", text)
    return tokens

train_df_tmp = pd.read_csv(train_path)
train_df_tmp["raw_tokens"]  = train_df_tmp["raw"].apply(parse_token_list)
train_df_tmp["norm_tokens"] = train_df_tmp["norm"].apply(parse_token_list)

mapping_counter = defaultdict(Counter)
for _, row in train_df_tmp.iterrows():
    raw_tokens  = row["raw_tokens"]
    norm_tokens = row["norm_tokens"]
    if len(raw_tokens) != len(norm_tokens):
        continue
    for raw_tok, norm_tok in zip(raw_tokens, norm_tokens):
        mapping_counter[raw_tok][norm_tok] += 1

# Tạo high_conf_lookup:
# Chỉ lấy token có:
# - confidence >= 0.95 (top1 chiếm >= 95%)
# - num_choices == 1 (chỉ có 1 nghĩa trong train)
# - total_count >= 5 (đủ dữ liệu để tin tưởng)
# - chưa có trong HARDCODE_DICT hoặc simple_lookup (tránh trùng)
high_conf_lookup = {}

for raw_tok, counter in mapping_counter.items():
    total_count  = sum(counter.values())
    top1_norm, top1_count = counter.most_common(1)[0]
    num_choices  = len(counter)
    confidence   = top1_count / total_count if total_count > 0 else 0

    if (
        confidence   >= 0.95 and
        num_choices  == 1    and
        total_count  >= 5    and
        raw_tok not in HARDCODE_DICT and
        raw_tok not in simple_lookup
    ):
        high_conf_lookup[raw_tok] = top1_norm

print(f"High-conf lookup entries: {len(high_conf_lookup)}")

# Xem thử một vài entry
sample_items = list(high_conf_lookup.items())[:20]
for raw, norm in sample_items:
    print(f"  {raw} → {norm}")

High-conf lookup entries: 1656
  thích → thích
  anh → anh
  mập → mập
  cứ → cứ
  ngây → ngây
  thơ → thơ
  thế → thế
  :)) → :))
  Nghê → Nghê
  xinh → xinh
  vậy → vậy
  mà → mà
  thấy → thấy
  bằng → bằng
  luôn → luôn
  Ê → Ê
  khóc → khóc
  được → được
  nào → nào
  má → má


In [26]:
# ============================================================
# 14. Hàm tính ERR
# ============================================================
def compute_err(val_df, pred_col):
    TP = FP = TN = FN = 0
    for _, row in val_df.iterrows():
        for raw, gold, pred in zip(
            row["raw_tokens"], row["norm_tokens"], row[pred_col]
        ):
            needs_norm        = (raw != gold)
            system_normalized = (pred != raw)
            correct           = (pred == gold)

            if not needs_norm and not system_normalized:
                TN += 1
            elif not needs_norm and system_normalized:
                FP += 1
            elif needs_norm and correct:
                TP += 1
            elif needs_norm and not correct:
                FN += 1

    ERR = (TP - FP) / (TP + FN) * 100
    return TP, FP, TN, FN, ERR

In [27]:
# ============================================================
# 15. Hàm normalize — V1: Qwen thuần
# ============================================================
def generate_v1(raw_token, raw_sentence):
    content = f"""Câu: "{raw_sentence}"
Token cần chuẩn hóa: "{raw_token}"
Hãy trả về dạng chuẩn của token đó trong ngữ cảnh câu trên."""

    response = ollama_client.chat(
        model="qwen2.5:14b",
        messages=[
            {
                "role": "system",
                "content": """Bạn là hệ thống chuẩn hóa văn bản tiếng Việt mạng xã hội.
Nhiệm vụ: chuyển token viết tắt về dạng chuẩn tiếng Việt.
Chỉ trả về đúng 1 token đã chuẩn hóa, không giải thích."""
            },
            {"role": "user", "content": content}
        ]
    )

    output = response["message"]["content"].strip()
    for sep in ["->", "→", ":"]:
        if sep in output:
            output = output.split(sep)[-1].strip()
    output = output.strip('"\'.,!? ')
    output = output.split("\n")[0].strip()
    return output

In [29]:
# ============================================================
# 16. Hàm normalize — V2: ViSoLex + Qwen
# ============================================================
def generate_v2(raw_token, raw_sentence):
    if raw_token in simple_lookup:
        return simple_lookup[raw_token]

    candidates = multi_lookup.get(raw_token, None)

    if candidates and raw_sentence:
        content = f"""Câu: "{raw_sentence}"
Token: "{raw_token}"
Các nghĩa có thể: {", ".join(candidates)}
Chỉ trả về đúng 1 từ phù hợp nhất với ngữ cảnh, không giải thích."""
    elif raw_sentence:
        content = f"""Câu: "{raw_sentence}"
Token cần chuẩn hóa: "{raw_token}"
Chỉ trả về đúng 1 token đã chuẩn hóa, không giải thích."""
    else:
        content = raw_token

    response = ollama_client.chat(
        model="qwen2.5:14b",
        messages=[
            {
                "role": "system",
                "content": """Bạn là hệ thống chuẩn hóa văn bản tiếng Việt mạng xã hội.
Nhiệm vụ: dựa vào ngữ cảnh câu, chuyển token viết tắt về dạng chuẩn tiếng Việt.
Chỉ trả về đúng 1 token đã chuẩn hóa, không giải thích."""
            },
            {"role": "user", "content": content}
        ]
    )

    output = response["message"]["content"].strip()
    for sep in ["->", "→", ":"]:
        if sep in output:
            output = output.split(sep)[-1].strip()
    output = output.strip('"\'.,!? ')
    output = output.split("\n")[0].strip()
    return output

In [30]:
# ============================================================
# 17. Hàm normalize — V3 (best): Hardcode + ViSoLex + Qwen
# ============================================================
def generate_v3(raw_token, raw_sentence):
    # Bước 1: Hardcode
    if raw_token in HARDCODE_DICT:
        return HARDCODE_DICT[raw_token]

    # Bước 2: Simple lookup
    if raw_token in simple_lookup:
        return simple_lookup[raw_token]

    # Bước 3: Multi lookup / OOV → Qwen
    candidates = multi_lookup.get(raw_token, None)

    if candidates and raw_sentence:
        content = f"""Câu: "{raw_sentence}"
Token: "{raw_token}"
Các nghĩa có thể: {", ".join(candidates)}
Chỉ trả về đúng 1 từ phù hợp nhất với ngữ cảnh, không giải thích."""
    elif raw_sentence:
        content = f"""Câu: "{raw_sentence}"
Token cần chuẩn hóa: "{raw_token}"
Chỉ trả về đúng 1 token đã chuẩn hóa, không giải thích."""
    else:
        content = raw_token

    response = ollama_client.chat(
        model="qwen2.5:14b",
        messages=[
            {
                "role": "system",
                "content": """Bạn là hệ thống chuẩn hóa văn bản tiếng Việt mạng xã hội.
Nhiệm vụ: dựa vào ngữ cảnh câu, chuyển token viết tắt về dạng chuẩn tiếng Việt.
Chỉ trả về đúng 1 token đã chuẩn hóa, không giải thích."""
            },
            {"role": "user", "content": content}
        ]
    )

    output = response["message"]["content"].strip()
    for sep in ["->", "→", ":"]:
        if sep in output:
            output = output.split(sep)[-1].strip()
    output = output.strip('"\'.,!? ')
    output = output.split("\n")[0].strip()
    return output

In [32]:
# ============================================================
# 17b. Hàm normalize — V4: High-conf lookup + Hardcode + ViSoLex + Qwen
# ============================================================
def generate_v4(raw_token, raw_sentence):
    # Bước 1: Hardcode
    if raw_token in HARDCODE_DICT:
        return HARDCODE_DICT[raw_token]

    # Bước 2: High-confidence lookup từ train
    if raw_token in high_conf_lookup:
        return high_conf_lookup[raw_token]

    # Bước 3: ViSoLex simple lookup
    if raw_token in simple_lookup:
        return simple_lookup[raw_token]

    # Bước 4: ViSoLex multi lookup / OOV → Qwen
    candidates = multi_lookup.get(raw_token, None)

    if candidates and raw_sentence:
        content = f"""Câu: "{raw_sentence}"
Token: "{raw_token}"
Các nghĩa có thể: {", ".join(candidates)}
Chỉ trả về đúng 1 từ phù hợp nhất với ngữ cảnh, không giải thích."""
    elif raw_sentence:
        content = f"""Câu: "{raw_sentence}"
Token cần chuẩn hóa: "{raw_token}"
Chỉ trả về đúng 1 token đã chuẩn hóa, không giải thích."""
    else:
        content = raw_token

    response = ollama_client.chat(
        model="qwen2.5:14b",
        messages=[
            {
                "role": "system",
                "content": """Bạn là hệ thống chuẩn hóa văn bản tiếng Việt mạng xã hội.
Nhiệm vụ: dựa vào ngữ cảnh câu, chuyển token viết tắt về dạng chuẩn tiếng Việt.
Chỉ trả về đúng 1 token đã chuẩn hóa, không giải thích."""
            },
            {"role": "user", "content": content}
        ]
    )

    output = response["message"]["content"].strip()
    for sep in ["->", "→", ":"]:
        if sep in output:
            output = output.split(sep)[-1].strip()
    output = output.strip('"\'.,!? ')
    output = output.split("\n")[0].strip()
    return output

In [34]:
# ============================================================
# 18. Pipeline runner chung
# ============================================================
def run_pipeline(val_df, generate_fn, pred_col, version_name):
    predictions = []

    for idx, row in val_df.iterrows():
        raw_tokens       = row["raw_tokens"]
        classifier_preds = row["classifier_preds"]
        raw_sentence     = " ".join(raw_tokens)

        pred_tokens = []
        for raw, label in zip(raw_tokens, classifier_preds):
            if label == 0:  # KEEP
                pred_tokens.append(raw)
            else:           # NEED_NORM → generate
                try:
                    normalized = generate_fn(raw, raw_sentence)
                    pred_tokens.append(normalized)
                except KeyboardInterrupt:
                    raise
                except Exception as e:
                    print(f"Lỗi token '{raw}': {e}")
                    pred_tokens.append(raw)  # fallback

        predictions.append(pred_tokens)
        if idx % 50 == 0:
            print(f"[{version_name}] Processed {idx}/{len(val_df)}")

    val_df[pred_col] = predictions
    val_df.to_csv(
        OUTPUT_PATH + f"val_predictions_{version_name}.csv",
        index=False, encoding="utf-8-sig"
    )
    print(f"[{version_name}] Done! Saved to val_predictions_{version_name}.csv")
    return val_df

In [35]:
# ============================================================
# 19. Chạy V1
# ============================================================
val_df = run_pipeline(val_df, generate_v1, "pred_tokens_v1", "V1")
TP, FP, TN, FN, ERR = compute_err(val_df, "pred_tokens_v1")
print(f"\n=== V1 (Qwen 14b thuần) ===")
print(f"TP: {TP}, FP: {FP}, TN: {TN}, FN: {FN}")
print(f"ERR: {ERR:.2f}%")

[V1] Processed 0/1050
Lỗi token 'sin': timed out
[V1] Processed 50/1050


KeyboardInterrupt: 

In [ ]:
# ============================================================
# 20. Chạy V2
# ============================================================
val_df = run_pipeline(val_df, generate_v2, "pred_tokens_v2", "V2")
TP, FP, TN, FN, ERR = compute_err(val_df, "pred_tokens_v2")
print(f"\n=== V2 (ViSoLex + Qwen 14b) ===")
print(f"TP: {TP}, FP: {FP}, TN: {TN}, FN: {FN}")
print(f"ERR: {ERR:.2f}%")

In [ ]:
# ============================================================
# 21. Chạy V3 (best)
# ============================================================
val_df = run_pipeline(val_df, generate_v3, "pred_tokens_v3", "V3")
TP, FP, TN, FN, ERR = compute_err(val_df, "pred_tokens_v3")
print(f"\n=== V3 - BEST (Hardcode + ViSoLex + Qwen 14b) ===")
print(f"TP: {TP}, FP: {FP}, TN: {TN}, FN: {FN}")
print(f"ERR: {ERR:.2f}%")

In [ ]:
# ============================================================
# 22. Chạy V4
# ============================================================
val_df = run_pipeline(val_df, generate_v4, "pred_tokens_v4", "V4")
TP, FP, TN, FN, ERR = compute_err(val_df, "pred_tokens_v4")
print(f"\n=== V4 (High-conf lookup + Hardcode + ViSoLex + Qwen 14b) ===")
print(f"TP: {TP}, FP: {FP}, TN: {TN}, FN: {FN}")
print(f"ERR: {ERR:.2f}%")

In [ ]:

# ============================================================
# 23. Tổng kết tất cả versions
# ============================================================
print("\n========== TỔNG KẾT ==========")
for pred_col, name in [
    ("pred_tokens_v1", "V1 Qwen 14b thuần              "),
    ("pred_tokens_v2", "V2 ViSoLex + Qwen 14b           "),
    ("pred_tokens_v3", "V3 Hardcode + ViSoLex + Qwen    "),
    ("pred_tokens_v4", "V4 High-conf + Hardcode + ViSoLex + Qwen"),
]:
    if pred_col in val_df.columns:
        TP, FP, TN, FN, ERR = compute_err(val_df, pred_col)
        print(f"{name} | TP:{TP} FP:{FP} FN:{FN} | ERR: {ERR:.2f}%")

In [ ]:
# ============================================================
# 24. Phân tích FN của V3 (best version)
# ============================================================
fn_cases = []
for _, row in val_df.iterrows():
    for raw, gold, pred in zip(
        row["raw_tokens"], row["norm_tokens"], row["pred_tokens_v3"]
    ):
        needs_norm = (raw != gold)
        correct    = (pred == gold)

        if needs_norm and not correct:
            fn_cases.append({
                "raw_token"       : raw,
                "gold_token"      : gold,
                "pred_token"      : pred,
                "in_hardcode"     : raw in HARDCODE_DICT,
                "in_simple"       : raw in simple_lookup,
                "in_multi"        : raw in multi_lookup,
            })

fn_df = pd.DataFrame(fn_cases)
fn_df["error_type"] = fn_df.apply(lambda r:
    "hardcode_wrong" if r["in_hardcode"] else
    "simple_wrong"   if r["in_simple"]   else
    "multi_wrong"    if r["in_multi"]    else
    "oov_wrong",
    axis=1
)

print(f"\nTổng FN (V3): {len(fn_df)}")
print("\nPhân loại FN:")
print(fn_df["error_type"].value_counts())

print("\nTop 20 token FN nhiều nhất:")
top_fn = (fn_df
    .groupby(["raw_token", "gold_token", "error_type"])
    .size().reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(20))
print(top_fn.to_string(index=False))

fn_df.to_csv(OUTPUT_PATH + "fn_analysis_v3.csv", index=False, encoding="utf-8-sig")
print("\nSaved fn_analysis_v3.csv")

# ============================================================
# 24. Save toàn bộ val_df với tất cả predictions
# ============================================================
val_df.to_csv(OUTPUT_PATH + "val_all_predictions.csv",
              index=False, encoding="utf-8-sig")
print("Saved val_all_predictions.csv")